# HR Analytics — Exploratory Data Analysis
**Employee Engagement, Burnout & Attrition**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

df = pd.read_csv('../data/raw/HR_Employee_Attrition.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Data Info & Missing Values
print('=== Dataset Info ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Attrition Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df['Attrition'].value_counts().plot(kind='bar', ax=axes[0], color=['#10b981','#ef4444'])
axes[0].set_title('Attrition Distribution')
axes[0].set_xlabel('')
df['Attrition'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['#10b981','#ef4444'])
axes[1].set_title('Attrition Rate')
plt.tight_layout()
plt.show()

In [ ]:
# Satisfaction Metrics Distribution
sat_cols = ['JobSatisfaction','EnvironmentSatisfaction','RelationshipSatisfaction','WorkLifeBalance','JobInvolvement']
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i, col in enumerate(sat_cols):
    df[col].value_counts().sort_index().plot(kind='bar', ax=axes[i], color='#667eea')
    axes[i].set_title(col.replace('Satisfaction','\nSat.'), fontsize=9)
    axes[i].set_xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap (numeric features)
num_cols = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(16, 12))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdYlGn', center=0, annot=False, linewidths=0.3)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Attrition by Department & Role
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
attrition_binary = (df['Attrition'] == 'Yes').astype(int)
dept_attr = df.groupby('Department').apply(lambda x: (x['Attrition']=='Yes').mean() * 100).sort_values()
dept_attr.plot(kind='barh', ax=axes[0], color='#ef4444', alpha=0.8)
axes[0].set_title('Attrition Rate by Department (%)')
axes[0].set_xlabel('Attrition %')
role_attr = df.groupby('JobRole').apply(lambda x: (x['Attrition']=='Yes').mean() * 100).sort_values()
role_attr.plot(kind='barh', ax=axes[1], color='#667eea', alpha=0.8)
axes[1].set_title('Attrition Rate by Job Role (%)')
axes[1].set_xlabel('Attrition %')
plt.tight_layout()
plt.show()

In [ ]:
# OverTime Impact
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
overtime_attr = df.groupby('OverTime').apply(lambda x: (x['Attrition']=='Yes').mean() * 100)
overtime_attr.plot(kind='bar', ax=axes[0], color=['#10b981','#ef4444'], rot=0)
axes[0].set_title('Attrition Rate by OverTime')
axes[0].set_ylabel('Attrition %')
for ot_val, color in [('No','#10b981'),('Yes','#ef4444')]:
    subset = df[df['OverTime']==ot_val]['MonthlyIncome']
    axes[1].hist(subset, bins=30, alpha=0.7, label=f'OT={ot_val}', color=color)
axes[1].set_title('Income Distribution by OverTime')
axes[1].set_xlabel('Monthly Income')
axes[1].legend()
df.boxplot(column='YearsAtCompany', by='Attrition', ax=axes[2])
axes[2].set_title('Tenure by Attrition')
axes[2].set_xlabel('Attrition')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# Outlier Detection
numeric_df = df.select_dtypes(include=[np.number])
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
for i, col in enumerate(numeric_df.columns[:15]):
    ax = axes[i // 5][i % 5]
    ax.boxplot(numeric_df[col].dropna())
    ax.set_title(col, fontsize=8)
    ax.set_xticks([])
plt.suptitle('Outlier Detection — Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()